In [ ]:
import json

from data import load_dataset
from segmentation_unet_train import run_unet_training

## 1) Load configuration from JSON

In [ ]:
with open('nucleolin_unet_config.json') as fd:
    config = json.load(fd)

dataset_paths = config['dataset_paths']
dataset_options = config['dataset_options']
network_options = config['network_options']

# network_options

## Alterative: Manually specify config

In [ ]:
dataset_paths = [
    {
        'base_path': '/Volumes/agl_data/AndreasMaiser/NSD/26AM06-02_1',
        'image_subfolder': 'patches_gfp+',
        'label_subfolder': 'patches-segmentation-threshold',
        'image_file_pattern': "*_ch0*.tif",
        'label_file_pattern': "*.tif"
    },
    {
        'base_path': '/Volumes/agl_data/AndreasMaiser/NSD/26AM06-02_2',
        'image_subfolder': 'patches_gfp+',
        'label_subfolder': 'patches-segmentation-threshold',
        'image_file_pattern': "*_ch1*.tif",
        'label_file_pattern': "*.tif"
    }
]

dataset_options = {
    'val_fraction': 0.15,
    'planeselect_min_labeled_pixels': 0,
    'planeselect_center_planes_fraction': 0.8,
    'plane_sliding_window': 1,
    'zero_channel_dropout_prob': 0.5,
    'normalization_strategy': 'per_image',
    'n_classes': 1,
    'sparse_labeling': True
}

network_options = {
    'unet_intermediate_channels': [64, 128, 128],
    'early_stop_patience': 50,
}

### Optional: save config to JSON

In [ ]:

# combine into single config to save
config = {
    'dataset_paths': dataset_paths,
    'dataset_options': dataset_options,
    'network_options': network_options
}

# with open('nucleolin_unet_config.json', 'w') as fd:
#     json.dump(config, fd, indent=4)

## 2) Load data

In [ ]:
dataset_train, dataset_val = load_dataset(dataset_paths, dataset_options)

In [ ]:
from matplotlib import pyplot as plt
from random import randint

img, mask = dataset_train[randint(0, len(dataset_train))]

fig, axs = plt.subplots(ncols=2)
axs[0].imshow(img.max(axis=0)[0]) # max-project along z-axis (NOTE: returns max val and index, we only want the max val)
axs[1].imshow(mask.squeeze())

len(dataset_train)

## 3) Run Training

In [ ]:
run_unet_training(dataset_train, dataset_val, dataset_options, network_options)

## Apply model to images

In [ ]:
from lightning.pytorch.utilities.model_summary import ModelSummary

net_inference =  LightningUNet.load_from_checkpoint('lightning_logs/version_34/checkpoints/epoch=19-step=5660.ckpt').eval()
# net_inference =  LightningUNet.load_from_checkpoint('/Users/david/Desktop/jurkat_nucleolin/unet_nucleolus_001/checkpoints/epoch=299-step=3300.ckpt').eval()

ModelSummary(net_inference, max_depth=3)

In [ ]:
test_file = '/Volumes/nn/Julia Vogtmann/Microscopy/26JV_018/tif/0001_ch0.tif'

# add two dummy dimensions (batch size, channels)
img = torch.from_numpy(imread(test_file)).float()[:, torch.newaxis, torch.newaxis]

# ALTERNATIVE with full loader (different batch size, etc.):
# img = torch.from_numpy(imread(test_file)).float()[:, torch.newaxis]
# predict_ds = torch.utils.data.TensorDataset(img)
# predict_loader = torch.utils.data.DataLoader(predict_ds, 1)


trainer = L.Trainer(enable_checkpointing=False, logger=False)
with torch.no_grad():
    pred = trainer.predict(net_inference, img)
    pred = torch.concat(pred)
    probs = torch.softmax(pred, 1)
    pred_labels = pred.argmax(1)

In [ ]:
import nd2

test_file = '/Volumes/agl_data/AndreasMaiser/NSD/26AM06-02_2/0010.nd2'
img = nd2.imread(test_file, dask=True, xarray=True)
img = img.isel(C=1).values
img = SparseLabeledImageDataset._normalize_intensities(img, NormalizationStrategy.PER_IMAGE)

sliding_window = 5
if sliding_window > 1:
    img = sliding_window_planewise_padded(img, sliding_window)
    img = torch.from_numpy(img).float()[:, torch.newaxis, :]
else:
    img = torch.from_numpy(img).float()[:, torch.newaxis, torch.newaxis]


trainer = L.Trainer(enable_checkpointing=False, logger=False)
with torch.no_grad():
    pred = trainer.predict(net_inference, img)
    pred = torch.concat(pred)

    if pred.shape[1] == 1:
        probs = torch.sigmoid(pred[:,0])
        pred_labels = probs > 0.5
    else:
        probs = torch.softmax(pred, 1)
        pred_labels = pred.argmax(1)

In [ ]:
import napari

if napari.current_viewer() is not None:
    napari.current_viewer().close()

viewer = napari.Viewer()
viewer.add_image(img[:,0,sliding_window//2])

# view predictions of a single class
# viewer.add_labels((pred_labels==2).int())

# ALTERNATIVE: all classes
viewer.add_labels(pred_labels.int())


viewer.add_image(probs)